> **Public copy.** This step reads the raw Jira export and the test trackers, which stay private, so it cannot be run from this repository. The outputs below are from the original run. After that run, product, vendor and system names and release numbers in the code, text and tables were replaced with the public codes used everywhere in this repository (Vendor A to E, releases such as 1.51, System tag 1 to 11 and so on), so every count is exactly as computed.


# 02 Releases: which release does each defect belong to?

**Goal:** give each defect a release, using what is written in Jira.

**Two release streams** (confirmed by Jagan, 20 Nov 2026):
- **Vendor A** main releases 1.28 to 1.5x, with increments (INC) and fix drops during testing (PTF). Vendor A. A number straight after a product name (Store management app v3.2.20, Vendor D v2.33.7) is that product's version, not a Vendor A release.
- **Vendor B** self-checkout releases (CI 4 to CI 12). Each version, such as CI 11.2, counts as its own release. Vendor B.

**Sources, from strongest to weakest:** the title (summary), labels, fixVersion, then environment and description, then comments.
The title comes first because the tester writes it when the defect is **found** ("[Release 1.39] ..."), while labels and fixVersion are often moved later to the release that will carry the fix. Notebook 04 tests this against the test-window dates: when the title and fixVersion disagree, the title's release contains the defect's date 21 times, fixVersion's only twice (23 Nov 2026).
Defects still without a release will be matched later by date, using the tracker test windows (notebook 04).

**Rule for several releases:** labels often build up as a defect is carried into later releases (for example CI 4 then CI 5).
So the **earliest** release named is taken as the release the defect was found in, and the number of releases it was carried through is kept as a feature.
Only the strong sources (fixVersion, labels, summary) count for "carried through", because comments often mention other releases in passing.

In [1]:
import sqlite3, sys
from pathlib import Path
import pandas as pd

# Works whether you run from the project folder or from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'scripts'))
WORK = ROOT / 'data' / 'private' / 'hindsight_work.sqlite'    # Jira export (read only)
CLEAN = ROOT / 'data' / 'private' / 'hindsight_clean.sqlite'  # our cleaned tables
pd.set_option('display.width', 200)
print('Reading from:', WORK)

Reading from: data/private/hindsight_work.sqlite


In [2]:
from hs_release import parse
con = sqlite3.connect(WORK)
with sqlite3.connect(CLEAN) as cc:
    defects = pd.read_sql('select key, summary, environment, created from defects', cc)
desc = pd.read_sql("select key, description from issues where issue_type='Defect'", con)
fv = pd.read_sql('select issue_key, name from fix_versions', con)
lb = pd.read_sql('select issue_key, label from labels', con)
cm = pd.read_sql('select issue_key, body from comments', con)
keys = set(defects.key)
print(len(defects), 'defects')

3152 defects


## 1. Quick test of the parser on real spellings

In [3]:
for t in ['1.34 pTF12 pt2', 'release1.52inc2ptf1', '1.43_inc1_ptf1', 'scoci11.1', '[Vendor B-11.2 Testing]',
          'CI 100 cash device', '3.2.20', 'release1.52inc2pf4', '1.39in3.1']:
    print(f'{t:28}', parse(t))

1.34 pTF12 pt2               [{'stream': 'Vendor A', 'main': '1.34', 'inc': None, 'ptf': 12}]
release1.52inc2ptf1          [{'stream': 'Vendor A', 'main': '1.52', 'inc': 2, 'ptf': 1}]
1.43_inc1_ptf1               [{'stream': 'Vendor A', 'main': '1.43', 'inc': 1, 'ptf': 1}]
scoci11.1                    [{'stream': 'Vendor B', 'main': 'CI 11', 'inc': None, 'ptf': None, 'ci_full': 'CI 11.1'}]
[Vendor B-11.2 Testing]        [{'stream': 'Vendor B', 'main': 'CI 11', 'inc': None, 'ptf': None, 'ci_full': 'CI 11.2'}]
CI 100 cash device           []
3.2.20                       []
release1.52inc2pf4           [{'stream': 'Vendor A', 'main': '1.52', 'inc': 2, 'ptf': 4}]
1.39in3.1                    [{'stream': 'Vendor A', 'main': '1.39', 'inc': 3, 'ptf': None}]


## 2. Find every release mention, source by source

In [4]:
rows = []
def add(source, rank, key, text, bare_ci=True):
    for h in parse(text, allow_bare_ci=bare_ci):
        rows.append({'key': key, 'source': source, 'rank': rank, **h})

for r in defects.itertuples():
    add('summary', 1, r.key, r.summary)               # written when the defect was found
    add('environment', 4, r.key, r.environment)
for r in lb.itertuples():
    if r.issue_key in keys: add('label', 2, r.issue_key, r.label)
for r in fv.itertuples():
    if r.issue_key in keys: add('fixVersion', 3, r.issue_key, r.name)   # often the release that carried the fix
for r in desc.itertuples():
    add('description', 4, r.key, r.description, bare_ci=False)
for r in cm.itertuples():
    if r.issue_key in keys: add('comment', 5, r.issue_key, r.body, bare_ci=False)

hits = pd.DataFrame(rows)
# Each Vendor B version is its own release (CI 11.1, CI 11.2 ...), as decided for Vendor B
ci = (hits.stream == 'Vendor B') & hits.ci_full.notna()
hits.loc[ci, 'main'] = hits.loc[ci, 'ci_full']
hits.groupby('source').key.nunique().sort_values(ascending=False)

source
summary        1126
label          1008
description     779
comment         714
fixVersion      358
environment     346
Name: key, dtype: int64

## 3. Pick one release per defect

For each defect, use the strongest source that names a release. Inside that source, take the earliest release (Vendor A by number, CI by number).

In [5]:
def order(main):
    if main.startswith('CI '):
        return (1, float(main.split()[1]))
    return (0, float(main.split('.')[1]))

def decide(g):
    best = g[g['rank'] == g['rank'].min()]
    mains = sorted(set(best.main), key=order)
    found = mains[0]
    row = best[best.main == found].iloc[0]
    strong = g[g['rank'] <= 3]                      # summary, label, fixVersion only
    named = strong if len(strong) else best
    all_mains = sorted(set(named.main), key=order)
    inc = best[best.main == found].inc.dropna()
    ptf = best[best.main == found].ptf.dropna()
    return pd.Series({
        'release_main': found,
        'stream': row.stream,
        'inc': int(inc.min()) if len(inc) else None,
        'ptf': int(ptf.min()) if len(ptf) else None,
        'release_source': row.source,
        'confidence': {1: 'high', 2: 'high', 3: 'high', 4: 'medium', 5: 'low'}[int(row['rank'])],
        'releases_named': ', '.join(all_mains),
        'n_releases_named': len(all_mains),
        'both_streams': named.stream.nunique() > 1,
    })

rel = hits.groupby('key').apply(decide).reset_index()
rel.confidence.value_counts()

confidence
high      1471
medium     202
low         82
Name: count, dtype: int64

In [6]:
summary = defects[['key']].merge(rel, on='key', how='left')
summary['confidence'] = summary.confidence.fillna('none yet')
print(summary.confidence.value_counts().to_string())
print()
print(f"Release found for {summary.release_main.notna().mean():.1%} of defects")

confidence
high        1471
none yet    1397
medium       202
low           82

Release found for 55.7% of defects


In [7]:
summary.groupby(['stream', 'release_main'], dropna=True).size().rename('defects').reset_index()

,stream,release_main,defects
0,Vendor A,1.28,1
1,Vendor A,1.29,10
2,Vendor A,1.30,2
3,Vendor A,1.31,5
4,Vendor A,1.32,7
5,Vendor A,1.33,76
6,Vendor A,1.34,102
7,Vendor A,1.35,1
8,Vendor A,1.36,84
9,Vendor A,1.37,155


## 3b. Build labels (3.2.x)

Labels such as 3.2.18, 3.2.20 and 3.2.21b are not Vendor A main releases. The trackers show what they are: in July and August 2019 they record "Vendor C 3.2.26 build" and "Vendor C management app v3.2.26". So 3.2.x numbers are **Vendor C app Management builds (Vendor C)**, and a number right after a product name is that product's version (agreed with Jagan, 23 Nov 2026).
Here we only keep the build number; notebook 04 places these defects in the Vendor C programmes.

In [8]:
builds = lb[lb.label.str.lower().str.fullmatch(r'3\.2\.\d+[a-z]?')]
builds = builds[builds.issue_key.isin(keys)].groupby('issue_key').label.agg(lambda s: ', '.join(sorted(set(s.str.lower()))))
summary['build_label'] = summary.key.map(builds)
print('defects with a 3.2.x build label:', int(summary.build_label.notna().sum()))

defects with a 3.2.x build label: 61


## 4. Checks

- **Carried over:** defects named in more than one release.
- **Both streams:** defects named in a Vendor A release and an Vendor B release.
- **By year:** coverage over time.

In [9]:
print('Named in more than one release:', int((summary.n_releases_named > 1).sum()))
print('Named in both streams:', int(summary.both_streams.fillna(False).sum()))
cov = summary.merge(defects[['key', 'created']], on='key')
cov['year'] = cov.created.str[:4]
cov.groupby('year').release_main.apply(lambda s: f"{s.notna().mean():.0%}")

Named in more than one release: 273
Named in both streams: 55


year
2016    92%
2017    44%
2018    59%
2019    75%
2020    50%
2021    58%
2022    36%
2023    45%
2024    50%
2025    65%
2026    38%
Name: release_main, dtype: str

## 5. Save `defect_release`

In [10]:
with sqlite3.connect(CLEAN) as cc:
    summary.to_sql('defect_release', cc, if_exists='replace', index=False)
    hits.to_sql('release_mentions', cc, if_exists='replace', index=False)
print('saved', len(summary), 'rows')

saved 3152 rows


## What is next

Defects marked "none yet" are matched by date in notebook 04, once the tracker test windows are updated to today (including the Vendor B releases).